# Developing chicken heart with anatomy-reviewed CytoBridge

This tutorial adds the GSE149457 D4/D7/D10/D14 chicken-heart series as the fifth package workflow. It starts from raw 10x counts, preserves the reviewed tissue geometry, and refuses anatomically mirrored coordinates. The known legacy D7 left/right reflection is repaired only through an explicit, recorded compatibility option.

**Learning goals**

- build the 3,550-spot input from raw integer counts;
- verify Atria/Valves and RV/LV anatomical orientation;
- plan or run graph fitting, training, and downstream analysis through the installed package;
- distinguish 50D gene/state velocity from direct 2D spatial velocity; and
- locate the standard growth, composition, communication, LR, gene-program, and figure outputs.

## Scientific contract

The public Visium slides are not passed to the generic spatial registration routine. The dataset-specific adapter selects the reviewed spots in a fixed order and copies the reviewed coordinates. It validates the vertical anatomy at every stage and the RV/LV side from D7 onward. D4 has no separate right/left ventricular label, so its contract is limited to Atria above Valves above Ventricle.

The human CellChatDB table is used only as an explicitly labeled conserved-symbol proxy; it is not described as a Gallus gallus-specific interaction database.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import CytoBridge as cb
from CytoBridge.workflow import (
    WorkflowOptions,
    build_workflow_plan,
    load_workflow_config,
    render_workflow_plan,
    run_workflow,
)

SEED = 42
DEVICE = 'cuda:0'
RUN_PREPROCESS = False
RUN_TRAIN_AND_DOWNSTREAM = False

## 1. Paths and explicit execution switches

Keep both switches false for a read-only walkthrough. Production training requires a CUDA environment with the package's `spatial`, `train`, and downstream dependencies.

In [ ]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'CytoBridge').is_dir():
    REPO_ROOT = REPO_ROOT.parent

RAW_DIR = REPO_ROOT / 'spatial_data' / 'GSE149457_RAW'
METADATA_H5AD = REPO_ROOT / 'spatial_data' / 'chicken_heart_spatial_merged_with_meta.h5ad'
REVIEWED_ALIGNMENT = REPO_ROOT / 'spatial_data' / 'heart_aligned_all_timepoints.h5ad'
PREPARED_ROOT = REPO_ROOT / 'tutorial_outputs' / 'chicken_heart_input'
PREPARED_H5AD = PREPARED_ROOT / 'chicken_heart_aligned_package.h5ad'
PREPARED_TABLE = PREPARED_ROOT / 'model_input.csv'
PREPARED_MANIFEST = PREPARED_ROOT / 'manifest.json'
WORKFLOW_ROOT = REPO_ROOT / 'tutorial_outputs' / 'chicken_heart'
GRAPH_DATABASE = REPO_ROOT / 'CytoBridge' / 'workflow_databases' / 'CellChatDB.ligrec.human.csv'

PREPARED_H5AD

## 2. Build the fixed-alignment input from raw counts

The explicit D7 flag is accepted only when the sole orientation failure is the known D7 RV/LV horizontal mirror. It cannot repair D10/D14, vertical inversions, missing labels, or arbitrary registrations. A corrected reviewed reference should be used without this flag.

In [ ]:
prepare_command = [
    sys.executable,
    str(REPO_ROOT / 'scripts' / 'prepare_chicken_heart_input.py'),
    '--raw-dir', str(RAW_DIR),
    '--metadata-h5ad', str(METADATA_H5AD),
    '--aligned-reference-h5ad', str(REVIEWED_ALIGNMENT),
    '--graph-database', str(GRAPH_DATABASE),
    '--repair-legacy-d7-left-right',
    '--output-h5ad', str(PREPARED_H5AD),
    '--output-table', str(PREPARED_TABLE),
    '--manifest', str(PREPARED_MANIFEST),
]
if RUN_PREPROCESS:
    PREPARED_ROOT.mkdir(parents=True, exist_ok=False)
    subprocess.run(prepare_command, cwd=REPO_ROOT, check=True)
else:
    print('Read-only mode. Command:', ' '.join(prepare_command))

## 3. Recompute the anatomical orientation checks

Validation is performed from the H5AD coordinates and region labels, not merely trusted from the manifest. D7, D10, and D14 must place Right ventricle to the right of Compact LV/septum.

In [ ]:
if PREPARED_H5AD.is_file():
    prepared = ad.read_h5ad(PREPARED_H5AD)
    fixed_contract = cb.pp.validate_prepared_chicken_heart_input(prepared)
    anatomy = fixed_contract['anatomical_orientation_qc']
    orientation_rows = []
    for stage, record in anatomy['timepoints'].items():
        orientation_rows.append({'stage': stage, **record['checks']})
    display(pd.DataFrame(orientation_rows).set_index('stage'))
    print(fixed_contract['coordinate_policy'], fixed_contract['coordinate_sha256'])
else:
    print('Run preprocessing or point PREPARED_H5AD to the signed prepared input.')

In [ ]:
if PREPARED_H5AD.is_file():
    region = prepared.obs['region'].astype(str).map(lambda value: ' '.join(value.split()))
    xy = np.asarray(prepared.obsm['spatial_aligned'])
    fig, axes = plt.subplots(1, 4, figsize=(13.2, 3.4), sharex=True, sharey=True)
    for ax, stage in zip(axes, ('D4', 'D7', 'D10', 'D14')):
        mask = prepared.obs['timepoint'].astype(str).eq(stage).to_numpy()
        categories = pd.Categorical(region[mask])
        ax.scatter(xy[mask, 0], xy[mask, 1], c=categories.codes, s=7, cmap='tab10')
        ax.set_title(stage)
        ax.set_aspect('equal')
        ax.set_axis_off()
    fig.suptitle('Anatomy-reviewed chicken-heart regions (x-right, y-up)')
    fig.tight_layout()

## 4. Plan the package-native graph, training, and downstream workflow

The workflow copies the validated fixed H5AD byte-for-byte, constructs one interaction graph per observed time, selects the learned edge-predictor threshold on its validation split, runs the packaged six-stage training plan, and then runs the standard downstream analyses.

In [ ]:
workflow_config, workflow_source = load_workflow_config('chicken_heart')
workflow_options = WorkflowOptions(
    input_h5ad=PREPARED_H5AD,
    output_dir=WORKFLOW_ROOT,
    device=DEVICE,
    train=True,
)
plan = build_workflow_plan(workflow_config, source=workflow_source, options=workflow_options)
print(render_workflow_plan(plan))

In [ ]:
if RUN_TRAIN_AND_DOWNSTREAM:
    if WORKFLOW_ROOT.exists():
        raise FileExistsError(f'Use a new immutable workflow root: {WORKFLOW_ROOT}')
    result = run_workflow(workflow_config, options=workflow_options)
    display(result)
else:
    print('Read-only mode: no graph fitting, training, or downstream job was started.')

## 5. Correct velocity interpretation

`X_latent` has 50 expression PCs. Gene/state velocity is therefore high-dimensional; scVelo may be used to project those state derivatives into a two-dimensional expression display. Spatial velocity is different: the model state begins with the two aligned spatial dimensions, so the package plots `velocity[:, :2]` directly on `spatial_aligned[:, :2]`. It must not pass that already-spatial vector through a second scVelo transition projection.

In [ ]:
summary_path = WORKFLOW_ROOT / 'downstream' / 'summary.json'
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    velocity = summary['analyses']['velocity']
    assert velocity['spatial_projection_mode'] == 'direct_model_spatial_vector'
    print('Velocity archive:', velocity['component_archive'])
    print('Spatial projection:', velocity['spatial_projection_mode'])
    display(pd.DataFrame({'figure': velocity['figures']}))
else:
    print('Downstream summary not present yet.')

## 6. Article/SI-style downstream bank

A complete full-model run writes observed/generated slice snapshots, a time mosaic, growth, composition, intrinsic/interaction/full spatial-velocity panels, sparse cell-type attention, strict LR tables, temporal gene programs, and 3D communication. These are generated by the same package APIs as the other four datasets. Dataset-specific manuscript assembly should consume these signed outputs rather than recompute trajectories in notebook cells.

In [ ]:
if summary_path.is_file():
    analyses = summary.get('analyses', {})
    rows = []
    for name, record in analyses.items():
        if isinstance(record, dict):
            rows.append({'analysis': name, 'status': record.get('status'), 'figure': record.get('figure')})
    display(pd.DataFrame(rows))
    communication = analyses.get('communication', {})
    ligand_receptor = analyses.get('ligand_receptor', {})
    print('Communication:', communication)
    print('Ligand-receptor:', ligand_receptor)

## 7. Benchmark scope

The within-dataset LOTO candidates are the interior stages D7 and D10; D4 and D14 are boundary stages and are not described as bracketed interpolation holdouts. Full-data reconstruction is a separate fitted-model diagnostic, not a held-out benchmark. A five-dataset benchmark report must retain dataset, split, method, target, transform, prediction, and source-roster provenance.

## Pitfalls to avoid

- Do not reuse the old D7-mirrored aligned H5AD.
- Do not run generic spatial registration on the fixed chicken-heart input.
- Do not call the human LR proxy a chicken-specific database.
- Do not project direct 2D spatial velocity with scVelo a second time.
- Do not mix an edge predictor or threshold from another coordinate/PCA space.
- Do not publish notebook previews as formal results; use a fresh immutable run root and its manifests.